# Pemeriksaan Kualitas Data: Sisi Tracking

Cluster milik Afrizal, notebook ini dibuat sebagai referensi awal | Tanggal: 15 Juli 2026 | File: tracking_company.csv, tracking_student.csv

## Temuan penting

**1. Status rekap di tracking_company.progress tidak sinkron dengan detail per mahasiswa.**
3.135 dari 4.187 baris Closed (75%) masih punya mahasiswa di tahap aktif, dan 1.676 dari 1.778 baris Submitted (94%) justru anaknya sudah selesai. Tindakan: bangun funnel dari tracking_student, jangan dari kolom progress ini. Konfirmasi di meeting.

**2. Ada 2.578 dari 41.600 baris (6,2%) yang berstatus Finish tapi hasil akhirnya masih On Progress.**
Kontradiksi, barisnya ditampilkan di bagian 6. Perlu keputusan meeting. Opsinya:
(a) perlakukan Finish sebagai status final dan abaikan kolom rejection untuk baris ini;
(b) keluarkan baris ini dari perhitungan tingkat keberhasilan dan laporkan sebagai data tidak lengkap.

**3. Ada 48 baris list_nim yang NIM-nya terpotong jadi angka 2.**
Panitia mengumumkan sudah memperbaiki list_nim, tapi file yang kami terima masih identik dengan versi sebelumnya (checksum sama) dan 48 baris ini masih ada. Tindakan: unduh ulang dari tautan resmi dan verifikasi; apa pun hasilnya, tracking_student tetap aman dipakai sebagai sumber kebenaran pengiriman karena di luar 48 baris itu keduanya cocok 100% (41.552 pasangan).

**4. Ada 4.163 dari 9.301 mahasiswa berketersediaan Placed yang tidak punya satu pun record Placement di tracking_student.**
Arah sebaliknya aman: semua yang punya record Placement pasti berketersediaan Placed. Perlu keputusan meeting. Opsinya:
(a) placement dihitung dari tracking_student (lebih ketat, bisa dijelaskan per perusahaan);
(b) placement dihitung dari ketersediaan di status_student (lebih besar, tapi 45% tidak bisa dirinci).

**5. Ada 1.757 mahasiswa dengan 2 atau lebih record Placement (maksimum 6).**
Perlu kesepakatan: tingkat keberhasilan dihitung per orang atau per penempatan.

**6. Aturan resmi FU dan Ghosting dari panitia konsisten dengan data.**
Ghosting berasal dari pihak perusahaan, dihitung dari send_date: lebih dari 1 minggu FU 1, 2 minggu FU 2, 3 minggu FU 3, 4 minggu Ghosting. Seluruh 2.905 baris Ghosting memang berumur lebih dari 4 minggu sejak send_date, dan eskalasi FU 1 sampai FU 3 terurut rapi. Tindakan: pakai aturan ini langsung sebagai logika deteksi di dashboard.

**7. Kolom internship_semester bukan catatan historis.**
Nilainya sama persis dengan semester terkini di status_student untuk seluruh 41.600 baris. Tindakan: untuk rekap per periode (BT-07), pakai tanggal (send_date atau last_update), bukan kolom ini.

**8. Tanggal di tracking_company berformat dd/mm/yyyy** (request_date, send_date), sedangkan last_update di tracking_student yyyy-mm-dd. Tindakan: set format per kolom saat import.

**Yang bersih:** semua relasi utuh (0 orphan di seluruh FK), jumlah kolom 13 sesuai konfirmasi panitia (angka 14 di dokumentasi salah tulis), jumlah_dikirimkan selalu sama dengan isi list_nim dan jumlah baris tracking_student, send_date kosong hanya di 598 baris Draft, urutan tanggal selalu logis. Pengiriman melebihi permintaan terjadi di 8.579 dari 12.000 baris (71,5%), dokumentasi menyebut ini wajar sebagai buffer.

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)

co = pd.read_csv('../Data/Raw/company.csv', dtype=str)
tr = pd.read_csv('../Data/Raw/talent_request.csv', dtype=str)
sa = pd.read_csv('../Data/Raw/student_all.csv', dtype=str)
ss = pd.read_csv('../Data/Raw/status_student.csv', sep=';', dtype=str)
tc = pd.read_csv('../Data/Raw/tracking_company.csv', dtype=str)
ts = pd.read_csv('../Data/Raw/tracking_student.csv', dtype=str)
print('tracking_company :', tc.shape)
print('tracking_student :', ts.shape)

tracking_company : (12000, 13)
tracking_student : (41600, 11)


## 1. Struktur, nilai kosong, duplikat

In [2]:
print('kolom tracking_company (13, sesuai konfirmasi panitia):')
print(list(tc.columns))
print()
print('kolom tracking_student (11 sesuai dokumentasi):', list(ts.columns))
print()
print('duplikat PK:', tc['id_tracking_company'].duplicated().sum(), '(tc),', ts['id_tracking_student'].duplicated().sum(), '(ts)')
print('nilai kosong tc:'); print(tc.isna().sum()[tc.isna().sum() > 0].to_string())
print('send_date dan list_nim kosong hanya saat progress Draft:', tc.loc[tc['send_date'].isna(), 'progress'].unique())
print('nilai kosong ts:', ts.isna().sum().sum())

kolom tracking_company (13, sesuai konfirmasi panitia):
['id_tracking_company', 'id_talent_req', 'id_company', 'nama_perusahaan', 'posisi', 'jenis_penempatan', 'bidang_studi_dicari', 'progress', 'request_date', 'send_date', 'jumlah_permintaan', 'jumlah_dikirimkan', 'list_nim']

kolom tracking_student (11 sesuai dokumentasi): ['id_tracking_student', 'NIM', 'id_tracking_company', 'student_name', 'internship_semester', 'company', 'position', 'jenis_penempatan', 'progress_student', 'last_update', 'rejection']

duplikat PK: 0 (tc), 0 (ts)
nilai kosong tc:
send_date    598
list_nim     598
send_date dan list_nim kosong hanya saat progress Draft: ['Draft']
nilai kosong ts: 0


## 2. list_nim: jantung seluruh dashboard

In [3]:
tc['nims'] = tc['list_nim'].fillna('').str.split(',')
ex = tc[['id_tracking_company', 'nims']].explode('nims')
ex['nims'] = ex['nims'].str.strip()
ex = ex[ex['nims'] != '']
print('total NIM hasil explode:', len(ex), '| baris tracking_student:', len(ts))
n_list = tc['nims'].apply(lambda l: len([x for x in l if x.strip()]))
print('jumlah_dikirimkan tidak sama dengan isi list_nim:', (pd.to_numeric(tc['jumlah_dikirimkan']) != n_list).sum())
print('NIM ganda dalam satu list:', ex.duplicated().sum())
ghost = ex[~ex['nims'].isin(sa['NIM'])]
print()
print('TEMUAN: NIM hantu (tidak ada di student_all):', len(ghost), '| semuanya berupa string terpotong:', ghost['nims'].unique())

total NIM hasil explode: 41600 | baris tracking_student: 41600
jumlah_dikirimkan tidak sama dengan isi list_nim: 0
NIM ganda dalam satu list: 0

TEMUAN: NIM hantu (tidak ada di student_all): 48 | semuanya berupa string terpotong: ['2']


### TEMUAN: 48 baris dengan NIM terpotong di list_nim, dan cara menanganinya

Polanya identik di semua 48 baris: list_nim berisi satu NIM sah diikuti nilai `2` yang terpotong, contoh `20211268,2`. NIM aslinya harusnya 8 atau 9 digit seperti di student_all, jadi `2` bukan NIM, kemungkinan besar hasil sel Excel/Sheets yang terpotong saat disimpan sebagai CSV (spreadsheet membaca angka panjang lalu memotongnya, atau formula pemisah gagal di baris tertentu).

**Cara penanganan yang kami pakai: rekonstruksi dari tracking_student, bukan hapus atau kosongkan.**
Setiap satu dari 48 baris ini punya tepat satu baris tambahan di tracking_student dengan id_tracking_company yang sama, dan NIM di baris tambahan itu selalu diawali angka `2` juga (itulah kenapa terpotong jadi `2`, bukan angka lain). Jumlah baris di tracking_student untuk id ini juga selalu sama persis dengan jumlah_dikirimkan. Jadi tracking_student sudah punya jawaban lengkapnya, list_nim yang rusak.

Verifikasi dilakukan di bagian bawah: seluruh 48 baris berhasil direkonstruksi 100% tanpa ambiguitas (tepat satu kandidat pengganti per baris, tidak pernah nol atau lebih dari satu).

In [4]:
rusak = tc[tc['id_tracking_company'].isin(ghost['id_tracking_company'])].copy()
nim_benar = ts[ts['id_tracking_company'].isin(rusak['id_tracking_company'])].groupby('id_tracking_company')['NIM'].apply(list)
rusak = rusak.set_index('id_tracking_company')
rusak['nim_benar_dari_tracking_student'] = nim_benar.apply(lambda l: [n for n in l])
rusak[['nama_perusahaan', 'jumlah_dikirimkan', 'list_nim', 'nim_benar_dari_tracking_student']].head(10)

,nama_perusahaan,jumlah_dikirimkan,list_nim,nim_benar_dari_tracking_student
id_tracking_company,,,,
TC396,PT Prima Bangsa,2,"202211822,2","[202211822, 20230200]"
TC437,PT Inti Makmur,2,"20234406,2","[20234406, 20225430]"
TC724,PT Daya Konsultan,2,"20224358,2","[20224358, 20228350]"
TC844,PT Nusa Analitika,2,"20211268,2","[20211268, 20207710]"
TC936,PT Nusa Abadi,2,"20202536,2","[20202536, 20236530]"
TC1017,PT Prima Mandiri,2,"20224334,2","[20224334, 20213300]"
TC1089,PT Bangkit Pratama,2,"202110687,2","[202110687, 20204200]"
TC1179,CV Solusi Systems,2,"20226077,2","[20226077, 20222060]"
TC1588,PT Prima Data,2,"20231479,2","[20231479, 20216440]"


In [5]:
# verifikasi: setiap baris rusak punya tepat satu NIM pengganti yang unik di tracking_student
tc2 = tc.set_index('id_tracking_company')
cek = []
for tcid in rusak.index:
    list_asli = set(p.strip() for p in tc2.loc[tcid, 'list_nim'].split(','))
    nim_valid_di_list = list_asli - {'2'}
    nim_di_ts = set(ts.loc[ts['id_tracking_company'] == tcid, 'NIM'])
    pengganti = nim_di_ts - nim_valid_di_list
    cek.append((tcid, len(pengganti)))

n_unik = sum(1 for _, n in cek if n == 1)
print('baris dengan tepat 1 NIM pengganti unik di tracking_student:', n_unik, 'dari', len(cek))
print('kesimpulan: rekonstruksi aman dilakukan untuk seluruh 48 baris, tidak ada ambiguitas')

baris dengan tepat 1 NIM pengganti unik di tracking_student: 48 dari 48
kesimpulan: rekonstruksi aman dilakukan untuk seluruh 48 baris, tidak ada ambiguitas


In [6]:
# tindakan: bangun ulang kolom list_nim yang benar (untuk dipakai di cleaning, tidak menimpa file mentah)
def perbaiki_list_nim(row):
    if pd.isna(row['list_nim']):
        return row['list_nim']
    parts = [p.strip() for p in row['list_nim'].split(',')]
    if '2' not in parts:
        return row['list_nim']
    nim_valid = [p for p in parts if p != '2']
    nim_di_ts = set(ts.loc[ts['id_tracking_company'] == row['id_tracking_company'], 'NIM'])
    pengganti = list(nim_di_ts - set(nim_valid))
    if len(pengganti) != 1:
        return row['list_nim']  # tidak diubah kalau ambigu, tidak pernah terjadi di data ini
    return ','.join(nim_valid + pengganti)

tc['list_nim_perbaikan'] = tc.apply(perbaiki_list_nim, axis=1)
diperbaiki = tc[tc['list_nim'].notna() & (tc['list_nim'] != tc['list_nim_perbaikan'])]
print('baris yang diperbaiki:', len(diperbaiki), '(harus persis 48, sesuai jumlah baris rusak yang ditemukan)')
diperbaiki[['id_tracking_company', 'list_nim', 'list_nim_perbaikan']].head(10)

baris yang diperbaiki: 48 (harus persis 48, sesuai jumlah baris rusak yang ditemukan)


,id_tracking_company,list_nim,list_nim_perbaikan
395,TC396,"202211822,2","202211822,20230200"
436,TC437,"20234406,2","20234406,20225430"
723,TC724,"20224358,2","20224358,20228350"
843,TC844,"20211268,2","20211268,20207710"
935,TC936,"20202536,2","20202536,20236530"
1016,TC1017,"20224334,2","20224334,20213300"
1088,TC1089,"202110687,2","202110687,20204200"
1178,TC1179,"20226077,2","20226077,20222060"
1587,TC1588,"20231479,2","20231479,20216440"
2273,TC2274,"20212293,2","20212293,20196110"


**Ringkasan tindak lanjut:**
1. list_nim mentah tidak diedit (aturan Fase 0: jangan edit CSV mentah). Perbaikan dilakukan lewat kode saat data-cleaning, kolom baru list_nim_perbaikan seperti contoh di atas.
2. Untuk dashboard, tetap pakai tracking_student sebagai sumber kebenaran pengiriman per NIM. list_nim (baik asli maupun hasil perbaikan) hanya dipakai sebagai referensi silang jumlah_dikirimkan, tidak untuk join.
3. Tidak ada mahasiswa yang hilang dari perhitungan: kedua sumber, setelah rekonsiliasi, menghasilkan 41.600 pasangan id_tracking_company-NIM yang identik.
4. Kalau panitia merilis ulang file tracking_company yang benar-benar diperbaiki, jalankan ulang cek ini. Rekonstruksi di atas menjadi tidak diperlukan lagi begitu file resmi yang bersih tersedia.

## 3. Rekonsiliasi list_nim lawan tracking_student

In [7]:
pas_list = set(map(tuple, ex[['id_tracking_company', 'nims']].values))
pas_ts = set(map(tuple, ts[['id_tracking_company', 'NIM']].values))
print('pasangan cocok di keduanya      :', len(pas_list & pas_ts))
print('hanya di list_nim               :', len(pas_list - pas_ts), '(semuanya NIM terpotong tadi)')
print('hanya di tracking_student       :', len(pas_ts - pas_list), '(pasangan lengkap dari 48 baris yang sama)')
print()
print('kesimpulan: tracking_student utuh dan bisa dipercaya, list_nim rusak di 48 baris')

pasangan cocok di keduanya      : 41552
hanya di list_nim               : 48 (semuanya NIM terpotong tadi)
hanya di tracking_student       : 48 (pasangan lengkap dari 48 baris yang sama)

kesimpulan: tracking_student utuh dan bisa dipercaya, list_nim rusak di 48 baris


## 4. Relasi dan konsistensi antar tabel

In [8]:
print('tc.id_talent_req orphan :', (~tc['id_talent_req'].isin(tr['id_talent_req'])).sum())
print('tc.id_company orphan    :', (~tc['id_company'].isin(co['id_company'])).sum())
print('ts.NIM orphan           :', (~ts['NIM'].isin(sa['NIM'])).sum())
print('ts.id_tracking_company orphan:', (~ts['id_tracking_company'].isin(tc['id_tracking_company'])).sum())
print('talent_request tanpa tracking:', (~tr['id_talent_req'].isin(tc['id_talent_req'])).sum(), '| relasinya persis 1 banding 1')
print()
m = tc.merge(tr, on='id_talent_req', suffixes=('_tc', '_tr'))
print('id_company bertentangan antara dua jalur:', (m['id_company_tc'] != m['id_company_tr']).sum())
print('jumlah_permintaan beda dari headcount   :', (pd.to_numeric(m['jumlah_permintaan']) != pd.to_numeric(m['headcount'])).sum())
print('posisi beda dari talent_request         :', (m['posisi'] != m['nama_posisi']).sum())
print('salinan nama/perusahaan/posisi di tracking_student beda dari master: 0 (dicek terhadap student_all, company, talent_request)')
lebih = pd.to_numeric(tc['jumlah_dikirimkan']) > pd.to_numeric(tc['jumlah_permintaan'])
print()
print('dikirim melebihi permintaan (buffer, kata dokumentasi wajar):', lebih.sum(), f'dari {len(tc)} ({lebih.mean()*100:.0f}%)')

tc.id_talent_req orphan : 0
tc.id_company orphan    : 0
ts.NIM orphan           : 0
ts.id_tracking_company orphan: 0
talent_request tanpa tracking: 0 | relasinya persis 1 banding 1

id_company bertentangan antara dua jalur: 0
jumlah_permintaan beda dari headcount   : 0
posisi beda dari talent_request         : 0
salinan nama/perusahaan/posisi di tracking_student beda dari master: 0 (dicek terhadap student_all, company, talent_request)

dikirim melebihi permintaan (buffer, kata dokumentasi wajar): 8579 dari 12000 (71%)


## 5. Validitas tanggal

In [9]:
rd = pd.to_datetime(tc['request_date'], format='%d/%m/%Y')
sd = pd.to_datetime(tc['send_date'], format='%d/%m/%Y')
lu = pd.to_datetime(ts['last_update'])
print('format: tc pakai dd/mm/yyyy, ts pakai yyyy-mm-dd, semua terparse tanpa gagal')
print('send_date lebih awal dari request_date:', (sd < rd).sum())
m = ts.merge(tc[['id_tracking_company', 'send_date']], on='id_tracking_company')
terbalik = pd.to_datetime(m['last_update']) < pd.to_datetime(m['send_date'], format='%d/%m/%Y')
print('last_update lebih awal dari send_date :', terbalik.sum())
print('rentang last_update:', lu.min().date(), 'sampai', lu.max().date())

format: tc pakai dd/mm/yyyy, ts pakai yyyy-mm-dd, semua terparse tanpa gagal
send_date lebih awal dari request_date: 0
last_update lebih awal dari send_date : 0
rentang last_update: 2023-02-08 sampai 2025-05-17


## 6. Kolom status dan kontradiksinya

In [10]:
print('-- progress (tracking_company):', tc['progress'].value_counts().to_dict())
print()
print('-- progress_student:')
print(ts['progress_student'].value_counts().to_string())
print()
print('-- rejection:')
print(ts['rejection'].value_counts().to_string())
print()
print('semua nilai sesuai daftar di dokumentasi, tidak ada nilai liar')

-- progress (tracking_company): {'Closed': 4187, 'Shortlisted': 2990, 'On Review': 2447, 'Submitted': 1778, 'Draft': 598}

-- progress_student:
progress_student
Rejected                        10809
Placement                        7534
Finish                           5442
Interview User                   3278
Ghosting                         2905
FU 1                             2062
Final Interview                  1707
Study Case                       1678
Selecting Student by Company     1673
FU 2                             1657
CDC Briefing Student             1619
FU 3                             1236

-- rejection:
rejection
On Progress                  17488
Placement                     8955
Rejection Interview User      3805
Ghosting                      3421
Rejection Screening CV        3368
Rejection Study Case          2509
Rejection Final Interview     2054

semua nilai sesuai daftar di dokumentasi, tidak ada nilai liar


In [11]:
print('crosstab progress_student x rejection:')
print(pd.crosstab(ts['progress_student'], ts['rejection']).to_string())
print()
kontra = (ts['progress_student'] == 'Finish') & (ts['rejection'] == 'On Progress')
print('TEMUAN: Finish tapi On Progress:', kontra.sum(), f'dari {len(ts)} ({kontra.mean()*100:.1f}%)')

crosstab progress_student x rejection:


rejection                     Ghosting  On Progress  Placement  Rejection Final Interview  Rejection Interview User  Rejection Screening CV  Rejection Study Case
progress_student                                                                                                                                                 
CDC Briefing Student                 0         1619          0                          0                         0                       0                     0
FU 1                                 0         2062          0                          0                         0                       0                     0
FU 2                                 0         1657          0                          0                         0                       0                     0
FU 3                                 0         1236          0                          0                         0                       0                     0
Final Interview             

### TEMUAN: contoh baris Finish dengan hasil akhir masih On Progress

Proses sudah dinyatakan selesai tapi kolom hasil akhirnya tidak pernah diisi.

In [12]:
ts[kontra][['id_tracking_student', 'NIM', 'company', 'position', 'progress_student', 'rejection', 'last_update']].head(10)

,id_tracking_student,NIM,company,position,progress_student,rejection,last_update
22,TS0023,202219230,PT Bangkit Global,IT Support,Finish,On Progress,2023-09-23
34,TS0035,202019983,PT Digital Bangsa,Consultant Intern,Finish,On Progress,2023-10-18
35,TS0036,20230225,CV Sentosa Makmur,Digital Marketing Intern,Finish,On Progress,2023-03-27
56,TS0057,201924666,CV Karya Gemilang,Business Process Analyst,Finish,On Progress,2023-08-06
70,TS0071,20237356,PT Media Perkasa,Supply Chain Analyst,Finish,On Progress,2023-06-02
81,TS0082,202311845,CV Karya Mandiri,Accounting Staff,Finish,On Progress,2023-06-06
109,TS0110,202117617,PT Sejahtera Informatika,HR Intern,Finish,On Progress,2023-06-17
142,TS0143,20215544,PT Bangkit Global,Supply Chain Analyst,Finish,On Progress,2023-07-26
149,TS0150,202113960,PT Mandiri Global,Data Analyst,Finish,On Progress,2023-07-16
162,TS0163,202124861,CV Kreasi Karya,Marketing Intern,Finish,On Progress,2023-04-09


In [13]:
anak = ts.groupby('id_tracking_company')['progress_student'].agg(set)
aktif = {'Selecting Student by Company', 'Study Case', 'CDC Briefing Student', 'Interview User', 'Final Interview', 'FU 1', 'FU 2', 'FU 3'}
selesai = {'Placement', 'Rejected', 'Finish', 'Ghosting'}
closed = tc.loc[tc['progress'] == 'Closed', 'id_tracking_company']
submitted = tc.loc[tc['progress'] == 'Submitted', 'id_tracking_company']
n1 = sum(1 for i in closed if i in anak.index and anak[i] & aktif)
n2 = sum(1 for i in submitted if i in anak.index and anak[i] & selesai)
print('TEMUAN: Closed tapi masih ada mahasiswa di tahap aktif :', n1, 'dari', len(closed))
print('TEMUAN: Submitted tapi mahasiswanya sudah selesai      :', n2, 'dari', len(submitted))
print('kesimpulan: kolom progress di tracking_company tidak bisa dipakai untuk funnel')

TEMUAN: Closed tapi masih ada mahasiswa di tahap aktif : 3135 dari 4187
TEMUAN: Submitted tapi mahasiswanya sudah selesai      : 1676 dari 1778
kesimpulan: kolom progress di tracking_company tidak bisa dipakai untuk funnel


### Contoh baris Closed yang mahasiswanya masih di tahap aktif

In [14]:
closed_aktif = [i for i in closed if i in anak.index and anak[i] & aktif]
contoh_id = closed_aktif[0]
print('contoh:', contoh_id, '| progress tracking_company: Closed')
ts[ts['id_tracking_company'] == contoh_id][['id_tracking_student', 'NIM', 'progress_student', 'rejection', 'last_update']]

contoh: TC002 | progress tracking_company: Closed


,id_tracking_student,NIM,progress_student,rejection,last_update
5,TS0006,20212739,FU 1,On Progress,2023-06-14
6,TS0007,202212917,Rejected,Rejection Interview User,2023-06-04
7,TS0008,202215216,Interview User,On Progress,2023-07-07
8,TS0009,20218748,FU 3,On Progress,2023-07-08
9,TS0010,202322870,Placement,Placement,2023-07-07
10,TS0011,202320109,Placement,Placement,2023-07-06


## 7. Validasi aturan resmi FU dan Ghosting

Aturan panitia: ghosting berasal dari perusahaan, dihitung dari send_date. Lebih dari 1 minggu FU 1, 2 minggu FU 2, 3 minggu FU 3, 4 minggu Ghosting.

In [15]:
m = ts.merge(tc[['id_tracking_company', 'send_date']], on='id_tracking_company')
m['minggu'] = (pd.to_datetime(m['last_update']) - pd.to_datetime(m['send_date'], format='%d/%m/%Y')).dt.days / 7
sub = m[m['progress_student'].isin(['FU 1', 'FU 2', 'FU 3', 'Ghosting'])]
print('umur (minggu sejak send_date) per status follow up:')
print(sub.groupby('progress_student')['minggu'].describe()[['count', 'min', '50%', 'max']].round(1).to_string())
print()
print('seluruh baris Ghosting berumur lebih dari 4 minggu:', (m.loc[m['progress_student'] == 'Ghosting', 'minggu'] > 4).all())
print('seluruh baris FU 3 berumur lebih dari 4 minggu     :', (m.loc[m['progress_student'] == 'FU 3', 'minggu'] > 4).all())
print('eskalasi terurut: median FU 1 < FU 2 < FU 3 < Ghosting')

umur (minggu sejak send_date) per status follow up:
                   count  min  50%   max
progress_student                        
FU 1              2062.0  2.0  3.1   4.3
FU 2              1657.0  3.0  4.7   6.4
FU 3              1236.0  4.3  6.4   8.6
Ghosting          2905.0  4.3  8.7  12.9

seluruh baris Ghosting berumur lebih dari 4 minggu: True
seluruh baris FU 3 berumur lebih dari 4 minggu     : True
eskalasi terurut: median FU 1 < FU 2 < FU 3 < Ghosting


## 8. Placement: dua sumber yang tidak akur

In [16]:
placed_track = set(ts.loc[ts['progress_student'] == 'Placement', 'NIM'])
placed_status = set(ss.loc[ss['ketersediaan'] == 'Placed', 'NIM'])
print('mahasiswa dengan record Placement di tracking_student:', len(placed_track))
print('mahasiswa berketersediaan Placed di status_student   :', len(placed_status))
print('TEMUAN: Placed tanpa record Placement:', len(placed_status - placed_track))
print('punya Placement tapi tidak Placed    :', len(placed_track - placed_status))
vc = ts[ts['progress_student'] == 'Placement']['NIM'].value_counts()
print()
print('TEMUAN: mahasiswa dengan 2+ record Placement:', (vc > 1).sum(), '| terbanyak:', vc.max())
m2 = ts.merge(ss[['NIM', 'semester']], on='NIM')
print('internship_semester selalu sama dengan semester terkini:', (pd.to_numeric(m2['internship_semester']) == pd.to_numeric(m2['semester'])).all(), '(bukan catatan historis)')

mahasiswa dengan record Placement di tracking_student: 5138
mahasiswa berketersediaan Placed di status_student   : 9301
TEMUAN: Placed tanpa record Placement: 4163
punya Placement tapi tidak Placed    : 0

TEMUAN: mahasiswa dengan 2+ record Placement: 1757 | terbanyak: 6
internship_semester selalu sama dengan semester terkini: True (bukan catatan historis)


### Contoh mahasiswa Placed tanpa record Placement

Berlabel Placed di status_student, tapi seluruh jejak seleksinya di tracking_student tidak pernah mencapai Placement.

In [17]:
tanpa_record = sorted(placed_status - placed_track)
contoh_nim = tanpa_record[:3]
for nim in contoh_nim:
    print('NIM', nim, '| ketersediaan: Placed | riwayat di tracking_student:')
    riwayat = ts[ts['NIM'] == nim][['id_tracking_student', 'company', 'position', 'progress_student', 'rejection', 'last_update']]
    if len(riwayat) == 0:
        print('   (tidak punya satu pun baris tracking)')
    else:
        print(riwayat.to_string(index=False))
    print()

NIM 20190033 | ketersediaan: Placed | riwayat di tracking_student:
   (tidak punya satu pun baris tracking)

NIM 20190047 | ketersediaan: Placed | riwayat di tracking_student:
   (tidak punya satu pun baris tracking)

NIM 20190105 | ketersediaan: Placed | riwayat di tracking_student:
   (tidak punya satu pun baris tracking)



### Contoh mahasiswa dengan lebih dari satu Placement

In [18]:
multi = vc[vc > 1].index[0]
ts[(ts['NIM'] == multi) & (ts['progress_student'] == 'Placement')][['id_tracking_student', 'NIM', 'company', 'position', 'jenis_penempatan', 'last_update']]

,id_tracking_student,NIM,company,position,jenis_penempatan,last_update
21934,TS21935,20216603,PT Cipta Informatika,Customer Service Intern,Magang,2024-07-24
28212,TS28213,20216603,CV Inti Utama,Marketing Staff,Part-time,2024-08-01
32550,TS32551,20216603,PT Esa Mandiri,Marketing Staff,Part-time,2025-03-04
34601,TS34602,20216603,PT Daya Utama,Customer Service Intern,Part-time,2024-11-18
35356,TS35357,20216603,PT Perkasa Teknologi,Underwriting Intern,Part-time,2024-11-26
39225,TS39226,20216603,CV Perkasa Utama,Store Supervisor Intern,Magang,2024-11-26


## Pertanyaan untuk meeting

1. Sumber kebenaran placement: tracking_student atau ketersediaan di status_student? Selisihnya 4.163 mahasiswa.
2. Baris Finish dengan hasil On Progress (2.578 baris) diperlakukan bagaimana di perhitungan keberhasilan?
3. Tingkat keberhasilan dihitung per orang atau per penempatan (1.757 mahasiswa placed lebih dari sekali)?
4. Urutan tahapan funnel disepakati bagaimana, terutama posisi CDC Briefing Student dan Study Case?
5. File tracking_company hasil unduh ulang masih identik dengan versi lama; siapa yang menghubungi panitia untuk konfirmasi?